<a href="https://www.kaggle.com/code/avikdas567/catechol-benchmark-hackathon?scriptVersionId=291452560" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/catechol-benchmark-hackathon/drfps_catechol_lookup.csv
/kaggle/input/catechol-benchmark-hackathon/catechol_full_data_yields.csv
/kaggle/input/catechol-benchmark-hackathon/catechol_single_solvent_yields.csv
/kaggle/input/catechol-benchmark-hackathon/fragprints_lookup.csv
/kaggle/input/catechol-benchmark-hackathon/acs_pca_descriptors_lookup.csv
/kaggle/input/catechol-benchmark-hackathon/utils.py
/kaggle/input/catechol-benchmark-hackathon/spange_descriptors_lookup.csv
/kaggle/input/catechol-benchmark-hackathon/smiles_lookup.csv


# Imports & Configuration

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from tqdm import tqdm

In [3]:
import sys
sys.path.append('/kaggle/input/catechol-benchmark-hackathon')

# Load Utilities

In [4]:
from utils import (
    load_data,
    load_features,
    generate_leave_one_out_splits,
    generate_leave_one_ramp_out_splits,
    INPUT_LABELS_NUMERIC,
    INPUT_LABELS_SINGLE_FEATURES,
    INPUT_LABELS_FULL_FEATURES,
    TARGET_LABELS,
)

# Feature Engineering Helpers

In [5]:
def build_solvent_features(solvent_names, lookup):
    features = []
    for s in solvent_names:
        if s in lookup.index:
            features.append(lookup.loc[s].values)
        else:
            features.append(np.zeros(lookup.shape[1]))
    return np.vstack(features)


def make_single_solvent_features(X, descriptor_lookup):
    num = X[INPUT_LABELS_NUMERIC].values
    solv = build_solvent_features(X["SOLVENT NAME"], descriptor_lookup)
    return np.hstack([num, solv])


def make_mixture_features(X, descriptor_lookup):
    num = X[INPUT_LABELS_NUMERIC].values
    solv_a = build_solvent_features(X["SOLVENT A NAME"], descriptor_lookup)
    solv_b = build_solvent_features(X["SOLVENT B NAME"], descriptor_lookup)
    frac_b = X["SolventB%"].values.reshape(-1, 1)
    mixed = (1 - frac_b) * solv_a + frac_b * solv_b
    return np.hstack([num, mixed])

# Load Data & Descriptors

In [6]:
# Load datasets
X_full, Y_full = load_data("full")
X_single, Y_single = load_data("single_solvent")

# Choose descriptor set (strong baseline)
descriptor_lookup = load_features("spange_descriptors")

print("Full data shape:", X_full.shape)
print("Single-solvent data shape:", X_single.shape)
print("Descriptor dimension:", descriptor_lookup.shape)

Full data shape: (1227, 5)
Single-solvent data shape: (656, 3)
Descriptor dimension: (26, 13)


# Feature Matrices

In [7]:
X_full_feat = make_mixture_features(X_full, descriptor_lookup)
X_single_feat = make_single_solvent_features(X_single, descriptor_lookup)

Y_full = Y_full.values
Y_single = Y_single.values

# Model Definition

In [8]:
class CatecholModel:
    def __init__(self):
        base = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )
        self.model = Pipeline([
            ("scaler", StandardScaler()),
            ("regressor", MultiOutputRegressor(base))
        ])

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        preds = self.model.predict(X)
        return np.clip(preds, 0.0, 1.0)

# Cross-Validation (Single-Solvent)

In [9]:
single_results = []

splits = list(generate_leave_one_out_splits(X_single, pd.DataFrame(Y_single)))

for fold, ((X_tr, Y_tr), (X_te, Y_te)) in enumerate(tqdm(splits)):
    Xtr = make_single_solvent_features(X_tr, descriptor_lookup)
    Xte = make_single_solvent_features(X_te, descriptor_lookup)

    model = CatecholModel()
    model.fit(Xtr, Y_tr.values)
    preds = model.predict(Xte)

    for i in range(len(preds)):
        single_results.append([
            fold,
            X_te.index[i],
            preds[i, 0],
            preds[i, 1],
            preds[i, 2],
        ])

100%|██████████| 24/24 [00:47<00:00,  1.98s/it]


# Cross-Validation (Mixture Solvents)

In [10]:
mixture_results = []

splits = list(generate_leave_one_ramp_out_splits(X_full, pd.DataFrame(Y_full)))

for fold, ((X_tr, Y_tr), (X_te, Y_te)) in enumerate(tqdm(splits)):
    Xtr = make_mixture_features(X_tr, descriptor_lookup)
    Xte = make_mixture_features(X_te, descriptor_lookup)

    model = CatecholModel()
    model.fit(Xtr, Y_tr.values)
    preds = model.predict(Xte)

    for i in range(len(preds)):
        mixture_results.append([
            fold,
            X_te.index[i],
            preds[i, 0],
            preds[i, 1],
            preds[i, 2],
        ])

100%|██████████| 13/13 [01:22<00:00,  6.36s/it]


# Create Submission

In [11]:
submission_rows = []

TASK_NAMES = ["Product 2", "Product 3", "SM"]

# Single-solvent results
for fold, row, p2, p3, sm in single_results:
    preds = [p2, p3, sm]
    for task, pred in zip(TASK_NAMES, preds):
        submission_rows.append([fold, row, task, pred])

# Mixture-solvent results
for fold, row, p2, p3, sm in mixture_results:
    preds = [p2, p3, sm]
    for task, pred in zip(TASK_NAMES, preds):
        submission_rows.append([fold, row, task, pred])

submission = pd.DataFrame(
    submission_rows,
    columns=["fold", "row", "task", "prediction"]
)

submission = submission.sort_values(["fold", "row", "task"]).reset_index(drop=True)

submission.to_csv("submission.csv", index=False)
submission.head(10)

,fold,row,task,prediction
0,0,0,Product 2,0.013931
1,0,0,Product 3,0.002863
2,0,0,SM,0.929517
3,0,1,Product 2,0.024726
4,0,1,Product 3,0.020250
5,0,1,SM,0.919079
6,0,2,Product 2,0.056218
7,0,2,Product 3,0.029021
8,0,2,SM,0.842716
9,0,3,Product 2,0.073734


In [12]:
print("Submission file written: submission.csv")
print("Rows:", len(submission))

Submission file written: submission.csv
Rows: 5649
